# CSWinUNETR on Google Colab Pro

Train and evaluate CSWinUNETR on FFHQ-Wrinkle. Run the cells **top to bottom**.

**Before you start:**
1. *Runtime → Change runtime type → **A100 GPU*** (or **L4 GPU** if no A100 is available). Turn on **High-RAM** if offered.
2. Upload your prepared dataset to Google Drive as **one zip file** (see `HOWTO.md`, Section 5). It must contain
   `masked_face_images/`, `weak_wrinkle_masks/` and `manual_wrinkle_masks/`.

**Every new session:** run Steps 1 and 2, restart the session, then Steps 3, 4 and 6. Training automatically **resumes** from the last saved epoch.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Get the code and install (needed once per session)

This downgrades Colab's torch/numpy to the versions the project pins.
**After this cell finishes, do *Runtime → Restart session*, then continue from Step 3.**

In [ ]:
import os
if not os.path.isdir('/content/CSWinUNETR'):
    !git clone https://github.com/tommy-mnan/CSWinUNETR.git /content/CSWinUNETR
%cd /content/CSWinUNETR
!pip install -q -e '.[train,eval]'
print('\nDone. Now: Runtime -> Restart session, then run Step 3.')

## Step 3 — Settings

Edit these once. **Keep `BATCH_SIZE`, `WORKERS`, `USE_CHECKPOINT` and `DATA_ROOT` the same for the whole run.**
`--resume` refuses to continue if any of them change, even if you later get a different GPU.

`BATCH_SIZE = 2` with gradient checkpointing is a safe choice that should fit on **both** L4 and A100,
so the run can continue on whichever GPU Colab gives you. Step 5 measures the real memory use, so check it before committing.

In [ ]:
%cd /content/CSWinUNETR
import numpy
if numpy.__version__ != '1.26.4':
    raise SystemExit(f'numpy {numpy.__version__} is loaded, but the project needs 1.26.4. '
                     'Restart the session (Restart button / Runtime -> Restart session), then run this cell again. '
                     'Do not rerun Step 2.')
import os, torch

DATA_ZIP   = '/content/drive/MyDrive/ffhq_wrinkle_data.zip'    # your dataset zip on Drive
DATA_ROOT  = '/content/data'                                   # local copy (fast); keep this path fixed
RUN_DIR    = '/content/drive/MyDrive/cswinunetr_runs/ffhq'     # checkpoints + logs (on Drive, survives disconnects)
TEST_DIR   = '/content/drive/MyDrive/cswinunetr_runs/ffhq-test'

BATCH_SIZE     = 2
USE_CHECKPOINT = True
WORKERS        = 8   # fixed number: --resume refuses if it changes between sessions

print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}, {props.total_memory / 2**30:.0f} GB | CPU cores: {os.cpu_count()} -> workers={WORKERS}')
else:
    print('No GPU! Runtime -> Change runtime type -> A100 / L4 GPU')

## Step 4 — Copy the dataset to local disk (every session)

Reading thousands of PNGs straight from Drive is slow, so unzip a local copy first.

In [ ]:
import glob
if not os.path.isdir(f'{DATA_ROOT}/masked_face_images'):
    !mkdir -p {DATA_ROOT} && unzip -q -o "{DATA_ZIP}" -d {DATA_ROOT}
    # If the zip had a top-level folder (e.g. data/), move its contents up one level.
    nested = glob.glob(f'{DATA_ROOT}/*/masked_face_images')
    if nested and not os.path.isdir(f'{DATA_ROOT}/masked_face_images'):
        top = os.path.dirname(nested[0])
        !mv "{top}"/* {DATA_ROOT}/ && rmdir "{top}"
!ls {DATA_ROOT}

In [ ]:
# Check the dataset: every split ID must exist in all three folders, and images must be 1024x1024.
from PIL import Image
from cswinunetr.datasets.ffhq_wrinkle import index_pngs, read_ids, validate_splits, SPLIT_DIR

print('Split sizes:', validate_splits())
folders = {name: index_pngs(f'{DATA_ROOT}/{sub}') for name, sub in
           [('images', 'masked_face_images'), ('textures', 'weak_wrinkle_masks'), ('masks', 'manual_wrinkle_masks')]}
problems = 0
for split in ('train', 'val', 'test'):
    ids = read_ids(SPLIT_DIR / f'{split}.txt')
    for name, files in folders.items():
        missing = [i for i in ids if i not in files]
        if missing:
            problems += 1
            print(f'[{split}] missing {len(missing)} {name}, e.g. {missing[:5]}')
sample = read_ids(SPLIT_DIR / 'train.txt')[0]
for name, files in folders.items():
    with Image.open(files[sample]) as im:
        print(f'{sample} {name}: size={im.size} mode={im.mode}')
print('Dataset OK' if not problems else 'FIX THE DATASET BEFORE TRAINING')

## Step 5 — Benchmark: memory and time estimate (optional but recommended)

Runs a few training steps on random data with your settings. It reports peak GPU memory, so you know the batch size fits,
and estimates hours per epoch, so you can plan your compute units.

In [ ]:
import time, torch
from cswinunetr import CSWinUNETR
from cswinunetr.utils.losses import dice_ce_loss
from schedulefree import AdamWScheduleFree

model = CSWinUNETR(4, 2, 2, use_checkpoint=USE_CHECKPOINT).cuda().train()
opt = AdamWScheduleFree(model.parameters(), lr=1e-3); opt.train()
x = torch.randn(BATCH_SIZE, 4, 1024, 1024, device='cuda')
y = (torch.rand(BATCH_SIZE, 1024, 1024, device='cuda') > 0.95).long()
torch.cuda.reset_peak_memory_stats()
times = []
for i in range(6):
    torch.cuda.synchronize(); t = time.perf_counter()
    opt.zero_grad(); dice_ce_loss(model(x), y).backward(); opt.step()
    torch.cuda.synchronize(); times.append(time.perf_counter() - t)
step = sorted(times[2:])[len(times[2:]) // 2]
per_img = step / BATCH_SIZE
peak = torch.cuda.max_memory_allocated() / 2**30
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Peak GPU memory: {peak:.1f} / {total:.0f} GB  (batch {BATCH_SIZE}, checkpointing={USE_CHECKPOINT})')
print(f'{per_img:.2f} s per training image -> ~{800 * per_img / 3600:.2f} h per epoch (GPU compute only)')
for epochs in (100, 300, 1000):
    print(f'  {epochs:4d} epochs ~ {epochs * 800 * per_img / 3600:6.1f} h')
print('Data loading/augmentation can add more time; compare with train_seconds in metrics.jsonl.')
del model, opt, x, y; torch.cuda.empty_cache()

## Step 6 — Train (starts fresh the first time, resumes automatically afterwards)

- The best model (by validation Dice) is saved to `RUN_DIR/best.pt`, and the resume state to `RUN_DIR/last.pt` after **every epoch**.
- If Colab disconnects, open a new session, run Steps 1 → 3 → 4, then this cell. It continues where it stopped.
- Keep this browser tab open: on Colab Pro, closing it can end the session.

In [ ]:
train_args = (f'--data-root {DATA_ROOT} --output-dir "{RUN_DIR}" --device cuda '
              f'--batch-size {BATCH_SIZE} --workers {WORKERS}' + (' --use-checkpoint' if USE_CHECKPOINT else ''))
if os.path.isfile(f'{RUN_DIR}/last.pt'):
    print('Found last.pt -> resuming'); train_args += ' --resume'
elif os.path.isdir(RUN_DIR) and os.listdir(RUN_DIR):
    raise SystemExit(f'{RUN_DIR} has files but no last.pt (the first epoch never finished). '
                     'Delete that folder or choose a new RUN_DIR, then rerun.')
else:
    print('Starting a new run')
!python cswinunetr/train.py {train_args}

## Monitor progress (run in a separate cell anytime)

In [ ]:
import json
rows = [json.loads(l) for l in open(f'{RUN_DIR}/metrics.jsonl')]
print(f"Epochs done: {rows[-1]['epoch']} | last loss: {rows[-1]['loss']:.4f} | last epoch time: {rows[-1]['epoch_seconds'] / 60:.1f} min")
vals = [(r['epoch'], r['val_dice']) for r in rows if 'val_dice' in r]
if vals:
    best = max(vals, key=lambda v: v[1])
    print(f'Best val Dice: {best[1]:.4f} at epoch {best[0]}')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot([r['epoch'] for r in rows], [r['loss'] for r in rows]); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
    ax[1].plot(*zip(*vals), marker='o'); ax[1].set_title('Validation Dice (EMA)'); ax[1].set_xlabel('epoch')
    plt.tight_layout(); plt.show()

## Step 7 — Evaluate on the test set

Run this after training finishes (or anytime, to evaluate the current `best.pt`). The output folder must be empty, so
delete `TEST_DIR` before rerunning.

In [ ]:
!python cswinunetr/inference.py \
  --checkpoint "{RUN_DIR}/best.pt" \
  --image-dir {DATA_ROOT}/masked_face_images \
  --texture-dir {DATA_ROOT}/weak_wrinkle_masks \
  --mask-dir {DATA_ROOT}/manual_wrinkle_masks \
  --ids-file cswinunetr/datasets/splits/ffhq/test.txt \
  --output-dir "{TEST_DIR}" \
  --all-metrics --device cuda

## View a prediction next to the ground truth

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
case = read_ids(SPLIT_DIR / 'test.txt')[0]   # change the index to view other cases
img  = np.array(Image.open(f'{DATA_ROOT}/masked_face_images/{case}.png').convert('RGB'))
gt   = np.array(Image.open(f'{DATA_ROOT}/manual_wrinkle_masks/{case}.png').convert('L')) > 0
pred = np.array(Image.open(f'{TEST_DIR}/{case}.png')) > 0
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, im, title in zip(ax, (img, gt, pred), ('Image', 'Ground truth', 'Prediction')):
    a.imshow(im, cmap=None if im.ndim == 3 else 'gray'); a.set_title(f'{title} ({case})'); a.axis('off')
plt.tight_layout(); plt.show()